In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

%matplotlib inline
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib.patches import Rectangle
import matplotlib.patheffects as path_effects

import os
from copy import copy

import analysis_tools as tool
from config import *

# Load Data

In [ ]:
grid_data = tool.load_grid_data()

In [ ]:
dts = pd.date_range(start="2021-07-10T00", end="2021-07-15T21", freq="3h")

base_dir = "/automount/agh/s6tifohr/july21_eval/data"

exp_name = "REA"
da_rea_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_rea_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_CTL"
da_ctl_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_ctl_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_WLT"
da_wlt_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_wlt_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

exp_name = "BLK_SAT"
da_sat_tp = tool.read_merged_var_det("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)
ds_sat_ens_tp = tool.read_merged_var_ens("tp", dts, f"{base_dir}/{exp_name}/merged/tp", accu=True)

# Precipitation Maps

In [ ]:
time_slice = slice(np.datetime64("2021-07-13T00"), np.datetime64("2021-07-15T12"))
timeframe = int((time_slice.stop - time_slice.start) / np.timedelta64(1, 'h'))

### Deterministic

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=3.2, h_pad=1.)

# Positions of subplots are required to align the colobars correctly:
pos_left = axs[1,0].get_position()  #(bottom) left subplot
pos_mid = axs[1,1].get_position()   #middle subplot
pos_right = axs[1,2].get_position() #subplot


# ICON-DREAM Reanalysis
ax = axs[0,0]
im = ax.tricontourf(grid_data["tri_26"], da_rea_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="ICON-DREAM")


# Saturation Run
ax = axs[0,1]
im = ax.tricontourf(grid_data["tri_26_red"], da_sat_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Wet Conditions")


# Wet - Ctrl
ax = axs[0,2]
mask_window = ((np.rad2deg(grid_data["grid_26_red"]["clon"]) >= PLOT_WINDOW[0]) & (np.rad2deg(grid_data["grid_26_red"]["clon"]) <= PLOT_WINDOW[1]) & 
               (np.rad2deg(grid_data["grid_26_red"]["clat"]) >= PLOT_WINDOW[2]) & (np.rad2deg(grid_data["grid_26_red"]["clat"]) <= PLOT_WINDOW[3]))

diff = da_sat_tp.sel(step=time_slice).sum(dim="step") - da_ctl_tp.sel(step=time_slice).sum(dim="step")
diff.data[~mask_window] = 0.

im = ax.tricontourf(grid_data["tri_26_red"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Wet - Control")


# Control Run
ax = axs[1,0]
im = ax.tricontourf(grid_data["tri_26_red"], da_ctl_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Control")


# Wilting Point Run
ax = axs[1,1]
im1 = ax.tricontourf(grid_data["tri_26_red"], da_wlt_tp.sel(step=time_slice).sum(dim="step"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Dry Conditions")

# Add ax for left colorbar
cbar1_ax = fig.add_axes([pos_left.x1 + (pos_mid.x0 - pos_left.x1 - 1.5 * pos_left.width)/2, #left
                         pos_left.y0 - 0.1,     #bottom
                         pos_left.width * 1.5,  #width
                         0.02])                 #height
fig.colorbar(im1, cax=cbar1_ax, orientation='horizontal', label=f"{timeframe}h precipitation sum in mm")


# Dry - Ctrl
ax = axs[1,2]
diff = da_wlt_tp.sel(step=time_slice).sum(dim="step") - da_ctl_tp.sel(step=time_slice).sum(dim="step")
diff.data[~mask_window] = 0.

im2 = ax.tricontourf(grid_data["tri_26_red"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Dry - Control")


# Add colorbar for right column
cbar2_ax = fig.add_axes([pos_right.x0, pos_right.y0 - 0.1, pos_right.width, 0.02])
fig.colorbar(im2, cax=cbar2_ax, orientation='horizontal', label="Precipitation difference in mm")


for ax, char in zip(axs.reshape(-1), ["a", "b", "c", "d", "e", "f"]):
    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.05, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])
    
plt.savefig(f'./figs/map_comp_det.png', dpi=600, bbox_inches='tight', format='png')
plt.show()

### Ensemble

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(10,8), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=3.2, h_pad=1.)

# Positions of subplots are required to align the colobars correctly:
pos_left = axs[1,0].get_position()  #(bottom) left subplot
pos_mid = axs[1,1].get_position()   #middle subplot
pos_right = axs[1,2].get_position() #subplot


# ICON-DREAM Reanalysis
ax = axs[0,0]
im = ax.tricontourf(grid_data["tri_28"], ds_rea_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="ICON-DREAM")


# Saturation Run
ax = axs[0,1]
im = ax.tricontourf(grid_data["tri_28"], ds_sat_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Wet Conditions")


# Wet - Ctrl
ax = axs[0,2]
mask_window = ((np.rad2deg(grid_data["grid_28"]["clon"]) >= PLOT_WINDOW[0]) & (np.rad2deg(grid_data["grid_28"]["clon"]) <= PLOT_WINDOW[1]) & 
               (np.rad2deg(grid_data["grid_28"]["clat"]) >= PLOT_WINDOW[2]) & (np.rad2deg(grid_data["grid_28"]["clat"]) <= PLOT_WINDOW[3]))

diff = ds_sat_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem") - ds_ctl_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem")
diff.data[~mask_window] = 0.

im = ax.tricontourf(grid_data["tri_28"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Wet - Control")


# Control Run
ax = axs[1,0]
im = ax.tricontourf(grid_data["tri_28"], ds_ctl_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Control")


# Wilting Point Run
ax = axs[1,1]
im1 = ax.tricontourf(grid_data["tri_28"], ds_wlt_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem"), levels=SUM_LVL, norm=SUM_NORM)
ax.set(title="Dry Conditions")

# Add ax for left colorbar
cbar1_ax = fig.add_axes([pos_left.x1 + (pos_mid.x0 - pos_left.x1 - 1.5 * pos_left.width)/2, #left
                         pos_left.y0 - 0.1,     #bottom
                         pos_left.width * 1.5,  #width
                         0.02])                 #height
fig.colorbar(im1, cax=cbar1_ax, orientation='horizontal', label=f"{timeframe}h precipitation sum in mm")


# Dry - Ctrl
ax = axs[1,2]
diff = ds_wlt_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem") - ds_ctl_ens_tp.sel(step=time_slice).sum(dim="step").median(dim="mem")
diff.data[~mask_window] = 0.

im2 = ax.tricontourf(grid_data["tri_28"], diff, levels=DIV_LVL, norm=DIV_NORM, cmap="coolwarm_r")
ax.set(title="Dry - Control")


# Add colorbar for right column
cbar2_ax = fig.add_axes([pos_right.x0, pos_right.y0 - 0.1, pos_right.width, 0.02])
fig.colorbar(im2, cax=cbar2_ax, orientation='horizontal', label="Precipitation difference in mm")


for ax, char in zip(axs.reshape(-1), ["a", "b", "c", "d", "e", "f"]):
    ax.set_extent(PLOT_WINDOW, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND)
    ax.add_feature(cfeature.OCEAN)
    ax.add_feature(cfeature.COASTLINE)

    # Create a Rectangle patch
    rectangle = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor='red', linewidth=2, fill=False)
    ax.add_patch(rectangle)

    ax.text(0.05, 0.93, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.savefig(f'./figs/map_comp_ens.png', dpi=600, bbox_inches='tight', format='png')
plt.show()

# Time Series

## Precipitation

### Deterministic

In [ ]:
# Compute time series of deterministic runs: 
cell_areas_focus = grid_data["area_26_red"].sel(cell=grid_data["focus_cells_26_red"])
ts_rea = (da_rea_tp * grid_data["area_26"]).sel(cell=grid_data["focus_cells_26"]).sum(dim="cell")
ts_ctl = (da_ctl_tp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
ts_wlt = (da_wlt_tp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")
ts_sat = (da_sat_tp.sel(cell=grid_data["focus_cells_26_red"]) * cell_areas_focus).sum(dim="cell")

In [ ]:
fig, ax = plt.subplots()

ax.plot(ts_rea["valid_time"], ts_rea, color="black", label="REA")
ax.plot(ts_ctl["valid_time"], ts_ctl, color="tab:blue", label="CTL")
ax.plot(ts_wlt["valid_time"], ts_wlt, color="tab:green", label="WLT")
ax.plot(ts_sat["valid_time"], ts_sat, color="tab:orange", label="SAT")

plt.legend()

ax.set(ylabel="Area precipitation in kg/h")

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_deterministic.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

### Ensemble

In [ ]:
# Compute time series of ensemble runs:
cell_areas_focus_ens = grid_data["area_28"].sel(cell=grid_data["focus_cells_28"])
ts_rea_ens = (ds_rea_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"]).sum(dim="cell")
ts_ctl_ens = (ds_ctl_ens_tp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
ts_wlt_ens = (ds_wlt_ens_tp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")
ts_sat_ens = (ds_sat_ens_tp.sel(cell=grid_data["focus_cells_28"]) * cell_areas_focus_ens).sum(dim="cell")

In [ ]:
fig, ax = plt.subplots()

ax.plot(ts_ctl_ens["valid_time"], ts_ctl_ens.median(dim="mem"), color="tab:blue", label="CTL")
ax.fill_between(ts_ctl_ens["valid_time"], ts_ctl_ens.quantile(0.25, dim="mem"), ts_ctl_ens.quantile(0.75, dim="mem"), color="tab:blue", alpha=0.3)

ax.plot(ts_wlt_ens["valid_time"], ts_wlt_ens.median(dim="mem"), color="tab:green", label="Dry")
ax.fill_between(ts_wlt_ens["valid_time"], ts_wlt_ens.quantile(0.25, dim="mem"), ts_wlt_ens.quantile(0.75, dim="mem"), color="tab:green", alpha=0.3)

ax.plot(ts_sat_ens["valid_time"], ts_sat_ens.median(dim="mem"), color="tab:orange", label="Wet")
ax.fill_between(ts_sat_ens["valid_time"], ts_sat_ens.quantile(0.25, dim="mem"), ts_sat_ens.quantile(0.75, dim="mem"), color="tab:orange", alpha=0.3)

ax.plot(ts_rea_ens["valid_time"], ts_rea_ens.median(dim="mem"), color="black", label="REA")

#plt.legend()

ax.set(ylabel="Area precipitation in kg/h")
ax.text(0.03, 0.96, "a", transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_ensemble.png', dpi=600, bbox_inches='tight', format='png')
plt.show()

In [ ]:
ctl_allsum = ts_ctl_ens.median(dim="mem").sum().item()
rea_allsum = ts_rea_ens.median(dim="mem").sum().item()
wlt_allsum = ts_wlt_ens.median(dim="mem").sum().item()
sat_allsum = ts_sat_ens.median(dim="mem").sum().item()

print(f"CTL (median) produces {(ctl_allsum - rea_allsum) / rea_allsum * 100:.02f}% more/less precipitation than DREAM (median).")
print(f"SAT (median) produces {(sat_allsum - ctl_allsum) / ctl_allsum * 100:.02f}% more/less precipitation than CTL (median).")
print(f"WLT (median) produces {(wlt_allsum - ctl_allsum) / ctl_allsum * 100:.02f}% more/less precipitation than CTL (median).")

## Soil Moisture

In [ ]:
dts = pd.date_range(start="2021-07-10T00", end="2021-07-15T21", freq="3h")

base_dir = "/automount/agh/s6tifohr/july21_eval/data"

mask = ((np.rad2deg(grid_data["grid_28"]["clon"]) >= PRUDENCE_REGIONS["ME"]["lon"].start) & 
        (np.rad2deg(grid_data["grid_28"]["clon"]) >= PRUDENCE_REGIONS["ME"]["lon"].stop) & 
        (np.rad2deg(grid_data["grid_28"]["clat"]) >= PRUDENCE_REGIONS["ME"]["lat"].start) & 
        (np.rad2deg(grid_data["grid_28"]["clat"]) >= PRUDENCE_REGIONS["ME"]["lat"].stop))

def compute_wso_iqr(exp_name, base_dir, dts, mask):
    """Just a helper function to limit memory usage."""
    ds_ens_wso = tool.read_merged_var_ens("W_SO", dts, f"{base_dir}/{exp_name}/merged/W_SO", accu=False)
    _ = ds_ens_wso.isel(cell=mask, depthBelowLandLayer=0).mean(dim="cell")
    
    pct_25 = _.quantile(0.25, dim="mem")
    pct_50 = _.quantile(0.5, dim="mem")
    pct_75 = _.quantile(0.75, dim="mem")

    return pct_25, pct_50, pct_75

In [ ]:
rea_25, rea_50, rea_75 = compute_wso_iqr("REA", base_dir, dts, mask)
ctl_25, ctl_50, ctl_75 = compute_wso_iqr("BLK_CTL", base_dir, dts, mask)
wlt_25, wlt_50, wlt_75 = compute_wso_iqr("BLK_WLT", base_dir, dts, mask)
sat_25, sat_50, sat_75 = compute_wso_iqr("BLK_SAT", base_dir, dts, mask)

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))

ax.plot(ctl_50["valid_time"], ctl_50, color="tab:blue", label="CTL")
ax.fill_between(ctl_50["valid_time"], ctl_25, ctl_75, color="tab:blue", alpha=0.3)

ax.plot(wlt_50["valid_time"], wlt_50, color="tab:green", label="Dry")
ax.fill_between(wlt_50["valid_time"], wlt_25, wlt_75, color="tab:green", alpha=0.3)

ax.plot(sat_50["valid_time"], sat_50, color="tab:orange", label="Wet")
ax.fill_between(sat_50["valid_time"], sat_25, sat_75, color="tab:orange", alpha=0.3)

ax.plot(rea_50["valid_time"], rea_50, color="black", label="REA")
ax.fill_between(rea_50["valid_time"], rea_25, rea_75, color="black", alpha=0.3)

plt.legend()

ax.set(ylabel=r"Average soil moisture in kg/m$^2$")
ax.text(0.03, 0.96, "b", transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_soil_moisture.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

### Precip & SM in one plot

In [ ]:
fig, axs = plt.subplots(2,1, figsize=(8,6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
fig.tight_layout(h_pad=0.8)

# Precip
ax = axs[0]
ax.plot(ts_ctl_ens["valid_time"], ts_ctl_ens.median(dim="mem"), color="tab:blue", label="CTL")
ax.fill_between(ts_ctl_ens["valid_time"], ts_ctl_ens.quantile(0.25, dim="mem"), ts_ctl_ens.quantile(0.75, dim="mem"), color="tab:blue", alpha=0.3)

ax.plot(ts_wlt_ens["valid_time"], ts_wlt_ens.median(dim="mem"), color="tab:green", label="Dry")
ax.fill_between(ts_wlt_ens["valid_time"], ts_wlt_ens.quantile(0.25, dim="mem"), ts_wlt_ens.quantile(0.75, dim="mem"), color="tab:green", alpha=0.3)

ax.plot(ts_sat_ens["valid_time"], ts_sat_ens.median(dim="mem"), color="tab:orange", label="Wet")
ax.fill_between(ts_sat_ens["valid_time"], ts_sat_ens.quantile(0.25, dim="mem"), ts_sat_ens.quantile(0.75, dim="mem"), color="tab:orange", alpha=0.3)

ax.plot(ts_rea_ens["valid_time"], ts_rea_ens.median(dim="mem"), color="black", label="REA")

ax.legend()

ax.set(ylabel="Area precipitation in kg/h")
ax.text(0.02, 0.96, "a", transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])


# SM
ax = axs[1]
ax.plot(ctl_50["valid_time"], ctl_50, color="tab:blue", label="CTL")
ax.fill_between(ctl_50["valid_time"], ctl_25, ctl_75, color="tab:blue", alpha=0.3)

ax.plot(wlt_50["valid_time"], wlt_50, color="tab:green", label="Dry")
ax.fill_between(wlt_50["valid_time"], wlt_25, wlt_75, color="tab:green", alpha=0.3)

ax.plot(sat_50["valid_time"], sat_50, color="tab:orange", label="Wet")
ax.fill_between(sat_50["valid_time"], sat_25, sat_75, color="tab:orange", alpha=0.3)

ax.plot(rea_50["valid_time"], rea_50, color="black", label="REA")
ax.fill_between(rea_50["valid_time"], rea_25, rea_75, color="black", alpha=0.3)

#plt.legend()

ax.set(ylabel=r"Average soil moisture in kg/m$^2$")
ax.text(0.02, 0.9, "b", transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])


plt.xticks(rotation=30)
plt.savefig(f'./figs/ts_precip_sm.png', dpi=600, bbox_inches='tight', format='png')
plt.show()

# Animation

In [ ]:
from PIL import Image
import glob, os

In [ ]:
exp1 = "rea"
exp2 = "ctl"
varname = "TQV"

day_range = range(1,15)

lvls_anim = [0,10,20,30,40,50,60]
norm_anim = colors.BoundaryNorm(lvls_anim, 256)

fnames = {"rea": [f"data/moisture_tracking/icon_dream/det/icon_R03B07_{varname.lower()}_202107{dd:02}.nc" for dd in day_range],
          "ctl": [f"data/moisture_tracking/blcklst_ctl/det/icon_R03B07_{varname.lower()}_202107{dd:02}.nc" for dd in day_range],
          "wlt": [f"data/moisture_tracking/blcklst_wlt/det/icon_R03B07_{varname.lower()}_202107{dd:02}.nc" for dd in day_range],
          "sat": [f"data/moisture_tracking/blcklst_sat/det/icon_R03B07_{varname.lower()}_202107{dd:02}.nc" for dd in day_range]}

In [ ]:
ds1 = xr.open_mfdataset(fnames[exp1])[varname]
ds2 = xr.open_mfdataset(fnames[exp2])[varname]

old_frames = glob.glob("./figs/frames/*")
for f in old_frames:
    os.remove(f)

for frame in range(len(ds1.time)):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3), 
                                subplot_kw={'projection': ccrs.PlateCarree()})
    fig.tight_layout()

    # First subplot
    ax1.set_extent([-30, 60, 30, 65], crs=ccrs.PlateCarree())   # set map extent
    contourf = ax1.contourf(ds1["lon"], ds1["lat"], ds1.isel(time=frame), 
                            levels=lvls_anim, norm=norm_anim, cmap='viridis',
                            transform=ccrs.PlateCarree())
    ax1.set_title(exp1.upper())
    ax1.add_feature(cfeature.COASTLINE)

    # Box for region in which observations are passive
    box_passive_1 = Rectangle((-10, 29), 50, 45, edgecolor="white", linewidth=2, fill=False, alpha=0.7, transform=ccrs.PlateCarree())
    ax1.add_patch(box_passive_1)

    # Second subplot
    ax2.set_extent([-30, 60, 30, 65], crs=ccrs.PlateCarree())
    contourf = ax2.contourf(ds2["lon"], ds2["lat"], ds2.isel(time=frame), 
                            levels=lvls_anim, norm=norm_anim, cmap='viridis',
                            transform=ccrs.PlateCarree())
    ax2.set_title(exp2.upper())
    ax2.add_feature(cfeature.COASTLINE)

    box_passive_2 = Rectangle((-10, 29), 50, 45, edgecolor="white", linewidth=2, fill=False, alpha=0.7, transform=ccrs.PlateCarree())
    ax2.add_patch(box_passive_2)

    # Colorbar
    plt.subplots_adjust(bottom=0.01)
    pos = ax2.get_position()
    cbar_ax = fig.add_axes([pos.x0, 0, pos.width, 0.05])
    cbar = plt.colorbar(contourf, cax=cbar_ax, orientation="horizontal")
    fig.text(pos.x0-0.09, pos.y0-0.1, r"TQV in $kg/m^2$", weight="bold")

    fig.text(pos.x0-0.3, pos.y0-0.1, f"{str(ds2.isel(time=frame)["time"].values)[:13]}", color="black", weight="bold")

    plt.savefig(f"figs/frames/frame_{frame:04d}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


# Compile frames into animation:
frames = []
for filename in sorted(glob.glob('figs/frames/*.png')):
    frames.append(Image.open(filename))

frames[0].save(f"./figs/animation_{varname}.gif", save_all=True, append_images=frames[1:], 
               duration=200, loop=0)

# Assimilation Region Showcase

In [ ]:
# This is not actually coinciding with the passive region, which is -10 to 40E and 30 to 75N, 
# but I used it to get a background in the image below  
mask_passive_region = ((np.rad2deg(grid_data["grid_det"]["clon"]) >= -20) & (np.rad2deg(grid_data["grid_det"]["clon"]) <= 50) & 
                       (np.rad2deg(grid_data["grid_det"]["clat"]) >= 20) & (np.rad2deg(grid_data["grid_det"]["clat"]) <= 85)).values
lons_pr, lats_pr = np.rad2deg(grid_data["grid_det"]["clon"][mask_passive_region]), np.rad2deg(grid_data["grid_det"]["clat"][mask_passive_region])

ii_passive_region = np.arange(len(mask_passive_region))[mask_passive_region]

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent([-30, 70, 25, 80], crs=ccrs.PlateCarree())

gl1 = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, alpha=0.15, zorder=10, color="black")
gl1.top_labels = False
gl1.right_labels = False

im = ax.tricontourf(grid_data["tri_28"], np.zeros(len(grid_data["area_28"])))
ax.text(50, 32, "EU Nest", color="white", weight="bold")

# Create a rectangle patch for the passive observation region:
box_passive = Rectangle((PASSIVE_REGION["x0"], PASSIVE_REGION["y0"]), PASSIVE_REGION["wx"], PASSIVE_REGION["wy"], edgecolor="purple", linewidth=2, fill=False)
ax.add_patch(box_passive)
ax.text(-9, 72, "Passive Observations", color="purple", weight="bold")

# Create a rectangle patch for the focus region
box_focus = Rectangle((FOCUS_REGION["x0"], FOCUS_REGION["y0"]), FOCUS_REGION["wx"], FOCUS_REGION["wy"], edgecolor="orange", linewidth=2, fill=False)
ax.add_patch(box_focus)
ax.text(11, 50, "Focus Region", color="orange", weight="bold")

ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)

plt.savefig("figs/demo_nest_passive_obs.png", dpi=600, bbox_inches='tight', format='png')
plt.show()

# Predictability

Get cell areas on lat-lon grid:

In [ ]:
R = 6371000  # Earth radius (m)
lat_spacing = 0.2
lon_spacing = 0.2

# Convert to radians
dlat = np.deg2rad(lat_spacing)
dlon = np.deg2rad(lon_spacing)

# Compute area
ll_grid = xr.open_dataset("./data/free_forecasts/2021071200/mem001/fc_ll_DOM02_0001.nc")

lat_rad = np.deg2rad(ll_grid["lat"])
area = R**2 * dlon * np.abs(np.sin(lat_rad + dlat/2) - np.sin(lat_rad - dlat/2))
area_fc = area + 0 * ll_grid["lon"] #hack to broadcast to lat/lon

Need mid-points of the focus region:

In [ ]:
mx = (FOCUS_REGION["x0"] + FOCUS_REGION["x1"])/2
my = (FOCUS_REGION["y0"] + FOCUS_REGION["y1"])/2
mt = np.datetime64("2021-07-14T00")

Precipitation sums in free forecasts

In [ ]:
init_sums = {} #sums for all initialization dates
dts_fc = pd.date_range("2021-07-01T00", "2021-07-12T00", freq="D")

for dt in dts_fc:
    print(dt)
    
    ens_sums = np.full(20, np.nan) #maximum sum for all members of one initialization date

    for mem in range(1,21):
        path = f"./data/free_forecasts/{dt.year}{dt.month:02}{dt.day:02}00/mem{mem:03}/"
        fnames = [path + fname for fname in sorted(os.listdir(path))[-97:]]
        da_mem = xr.open_mfdataset(fnames)["tot_prec"].diff(dim="time")        

        da_mem = da_mem.sel(lon=slice(FOCUS_REGION["x0"] - 3, FOCUS_REGION["x1"] + 3), 
                            lat=slice(FOCUS_REGION["y0"] - 3, FOCUS_REGION["y1"] + 3), 
                            time=slice(np.datetime64("2021-07-12"), np.datetime64("2021-07-16")), 
                            drop=True) * area_fc

        rolling_sum = da_mem.rolling(lon=int(FOCUS_REGION["wx"]/lon_spacing), lat=int(FOCUS_REGION["wy"]/lat_spacing), time=48, center=True).sum().compute()
        cutout = rolling_sum.sel(lon=slice(mx-1, mx+1), lat=slice(my-1, my+1), time=slice(mt-np.timedelta64(12,"h"), mt+np.timedelta64(12,"h")))

        # If the indeces are needed:
        #ii_max = cutout.argmax(..., skipna=True)
        #cutout_max = cutout.isel(time=ii_max["time"], lat=ii_max["lat"], lon=ii_max["lon"])
        #x0_max, y0_max = cutout_max["lon"] - wx/2, cutout_max["lat"] - wy/2

        ens_sums[mem-1] = cutout.max(skipna=True)

    init_sums[dt] = ens_sums

Precipitation sums in storyline scenarios:

In [ ]:
time_slice = slice(np.datetime64("2021-07-13T00"), np.datetime64("2021-07-15T00"))

rea_sums = (ds_rea_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])
ctl_sums = (ds_ctl_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])
sat_sums = (ds_sat_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])
wlt_sums = (ds_wlt_ens_tp * grid_data["area_28"]).sel(cell=grid_data["focus_cells_28"], step=time_slice).sum(dim=["cell", "step"])

Amount of moisture backtracked:

In [ ]:
fnames_rea = [f"data/moisture_tracking/REA/output/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_rea = xr.open_mfdataset(fnames_rea)
ds_track_rea = ds_track_rea.reindex(latitude=list(reversed(ds_track_rea["latitude"])))

R = 6371000  # Earth radius (m)
lat_spacing = 0.15
lon_spacing = 0.15

# Convert to radians
dlat = np.deg2rad(lat_spacing)
dlon = np.deg2rad(lon_spacing)

# Compute area
ll_grid = xr.open_dataset("data/moisture_tracking/REA/expanded_region.nc")

lat_rad = np.deg2rad(ll_grid["latitude"])
area = R**2 * dlon * np.abs(np.sin(lat_rad + dlat/2) - np.sin(lat_rad - dlat/2))
area_mt = area + 0 * ll_grid["longitude"] #hack to broadcast to lat/lon

sum_tagged = (area_mt * ds_track_rea["tagged_precip"]).sum().values
ts_evap = (area_mt * ds_track_rea["e_track"]).sum(dim=["latitude", "longitude"])
ts_evap = ts_evap[::-1].cumsum()[::-1]
ts_loss = (area_mt * ds_track_rea["losses"]).sum(dim=["latitude", "longitude"])

In [ ]:
fig, ax1 = plt.subplots()


# Plot share of evaporation:
ax1.plot(range(1,13), ts_evap[:12] / sum_tagged)
ax1.set_ylim(0,1)
ax1.set_yticks(np.arange(0, 1.1, 0.1))
ax1.set(ylabel="Share of precipitation tracked to source")

# x orientation:
ax1.tick_params(axis="x", labelrotation=45)


# Add explaination to FC init.
x1, x2 = 1, 12
y1, y2 = -0.15, -0.18

ax1.vlines([x1, x2], y1-0.02, y1+0.02, colors='k', lw=1.5,
          transform=ax1.get_xaxis_transform(), clip_on=False)
ax1.plot([x1, x2], [y1, y1], 'k-', lw=1.5, clip_on=False,
        transform=ax1.get_xaxis_transform())

ax1.text((x1+x2)/2, y2, "Forecast initialized on", ha="center", va="top",
        transform=ax1.get_xaxis_transform())


# Plot simulated precipitation sums:
ax2 = ax1.twinx()

x = [init_sums[dt] for dt in init_sums.keys()] + [rea_sums.values, ctl_sums.values]#, wlt_sums.values, sat_sums.values]
tick_labels = [f"{dt.day}.{dt.month}." for dt in dts_fc] + ["DREAM", "LDA Ctl"]#, "LDA Dry", "LDA Wet"] #tick labels are overriden by boxplot

bp = ax2.boxplot(x, tick_labels=tick_labels)
ax2.set(ylabel="Area Precipitation in kg")
ax2.set_ylim(0, 2.1e13)

# Highlight ICON-DREAM:
idx, lw = -2, 1.5
bp["boxes"][idx].set_linewidth(lw)
bp["whiskers"][2*idx].set_linewidth(lw)
bp["whiskers"][2*idx+1].set_linewidth(lw)
bp["caps"][2*idx].set_linewidth(lw)
bp["caps"][2*idx+1].set_linewidth(lw)


plt.savefig(f'./figs/improvement_demo.png', dpi=300, bbox_inches='tight', format='png')
plt.show()

# Moisture Tracking

In [ ]:
fnames_rea = [f"data/moisture_tracking/REA/output/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_rea = xr.open_mfdataset(fnames_rea)
ds_track_rea = ds_track_rea.reindex(latitude=list(reversed(ds_track_rea["latitude"])))

fnames_ctl = [f"data/moisture_tracking/BLK_CTL/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_ctl = xr.open_mfdataset(fnames_ctl)
ds_track_ctl = ds_track_ctl.reindex(latitude=list(reversed(ds_track_ctl["latitude"])))

fnames_wlt = [f"data/moisture_tracking/BLK_WLT/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_wlt = xr.open_mfdataset(fnames_wlt)
ds_track_wlt = ds_track_wlt.reindex(latitude=list(reversed(ds_track_wlt["latitude"])))

fnames_sat = [f"data/moisture_tracking/BLK_SAT/output/det/backtrack_2021-07-{dd:02}T00-00.nc" for dd in range(1,15)]
ds_track_sat = xr.open_mfdataset(fnames_sat)
ds_track_sat = ds_track_sat.reindex(latitude=list(reversed(ds_track_sat["latitude"])))

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10,5), subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout(w_pad=1, h_pad=1)

ax = axs[0,0]
im = ax.contourf(ds_track_rea["longitude"], ds_track_rea["latitude"], ds_track_rea["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="ICON-DREAM")

ax = axs[0,1]
im = ax.contourf(ds_track_sat["longitude"], ds_track_sat["latitude"], ds_track_sat["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="Wet Conditions")

ax = axs[1,0]
im = ax.contourf(ds_track_ctl["longitude"], ds_track_ctl["latitude"], ds_track_ctl["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="Control")

ax = axs[1,1]
im = ax.contourf(ds_track_wlt["longitude"], ds_track_wlt["latitude"], ds_track_wlt["e_track"].sum(dim="time"), levels=W2L_LVL, norm=W2L_NORM, cmap="Blues")
ax.set(title="Dry Conditions")


for ax, char in zip(axs.reshape(-1), ["a", "b", "c", "d"]):
    ax.add_feature(cfeature.COASTLINE)
    ax.set_extent([-80, 30, 25, 65], crs=ccrs.PlateCarree())    #Bounds: West, East, South, North

    ax.text(0.05, 0.86, char, transform=ax.transAxes,
        ha="center", va="center", fontweight="bold", 
        color='white', zorder=1,
        path_effects=[path_effects.withStroke(linewidth=3, foreground='black')])

plt.subplots_adjust(bottom=0.08)
cbar_ax = fig.add_axes([0.35, 0.02, 0.6, 0.03])
cbar = fig.colorbar(im, cax=cbar_ax, orientation="horizontal")

fig.text(0.15, 0.02, "Tagged Evaporation in mm")

plt.savefig(f'./figs/tracking_comp_det.png', dpi=600, bbox_inches='tight', format='png')
plt.show()